# Day 1 — What Makes an "Agent"? (Loop + Tools)

---

Welcome to Section 7. Before we dive into frameworks, one 75-min class to align on **what an agent actually is** — since much of the machinery (function calling, tool schemas, streaming) was already introduced in earlier sections.

You'll answer three questions today:

1. What's the difference between an LLM call and an *agent*?
2. What does the agent **loop** look like in code?
3. Why do we quickly outgrow hand-rolled loops and reach for **LangGraph** (Day 2)?

**Prerequisites from earlier sections:**
- **Section 4 Day 5.1** — tool schemas + function calling with `tool_choice="auto"`
- **Section 4 Day 5.2** — streaming responses
- **Section 6 Day 5** — prompt injection defenses (relevant for tool arguments)


## 1. Agent = LLM in a loop with tools

A regular LLM call:

```
   question  ->  [LLM]  ->  answer     (done)
```

An agent:

```
   question  ->  [LLM]  ->  "I need to search the web for X"
                                │
                                ▼
                        [call web_search tool]
                                │
                                ▼
                         result: "..."
                                │
                                ▼
                             [LLM]  ->  "Now I need to fetch this URL"
                                │
                                ▼
                        [call fetch tool]
                                │
                                ▼
                             [LLM]  ->  answer  (done)
```

The LLM keeps **thinking, acting, observing** until it decides it has the answer. That's the loop.

Three ingredients:
- **Perception** — the LLM reads the current state (question + past steps)
- **Planning + Action** — decides "call tool X with args Y" or "I'm done"
- **Observation** — the tool result feeds back into the next thought


## 2. The ReAct pattern (Reason + Act)

**ReAct** is the classic name for this loop pattern. The LLM alternates between:
- **Reason** — a "thought" cell (what should I do next?)
- **Act** — call a tool with structured arguments
- **Observe** — the tool result is inserted into context, then reason again

With modern function-calling APIs, you don't need to prompt-hack the "Thought / Action / Observation" format anymore — the model outputs structured `tool_calls` in its response. But the loop shape is the same, and job interviews still ask about ReAct by name.

**Sibling patterns to know (mention only):**
- **Plan-and-Execute** — LLM writes a full multi-step plan up front, then executes
- **Reflexion** — after failing, LLM writes a "lesson learned" and retries
- **Chain-of-Thought (CoT)** — just "think step-by-step" reasoning, no tools

95% of production agents in 2026 are ReAct or LangGraph-orchestrated variants of it.


## 3. Setup + real tools

You saw the calculator + weather tool in Section 4 Day 5.1. Today let's use tools an agent actually reaches for in the wild: **web search**, **HTTP fetch**, and a safe **calculator**.


In [ ]:
!pip install together tavily-python httpx python-dotenv --quiet

In [ ]:
import ast, json, operator, os
from dotenv import load_dotenv
from together import Together
import httpx

load_dotenv()
assert os.getenv("TOGETHER_API_KEY"), "Set TOGETHER_API_KEY in .env"
llm = Together()
MODEL = "openai/gpt-oss-20b"


# --- Safe calculator (AST-based, no eval) ---
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub,
        ast.Mult: operator.mul, ast.Div: operator.truediv,
        ast.Pow: operator.pow, ast.USub: operator.neg}
def _eval(n):
    if isinstance(n, ast.Num):    return n.n
    if isinstance(n, ast.BinOp):  return _OPS[type(n.op)](_eval(n.left), _eval(n.right))
    if isinstance(n, ast.UnaryOp): return _OPS[type(n.op)](_eval(n.operand))
    raise ValueError("bad expr")
def calc(expr: str) -> str:
    try:    return str(_eval(ast.parse(expr, mode="eval").body))
    except Exception as e: return f"error: {e}"


# --- Web search: Tavily if key present, else a mock so the demo still runs ---
def web_search(query: str) -> str:
    if not os.getenv("TAVILY_API_KEY"):
        return f"[MOCK] top result for {query!r}: (set TAVILY_API_KEY for real search)"
    from tavily import TavilyClient
    tv = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
    r = tv.search(query, max_results=3)
    return "\n".join(f"- {x['title']}: {x['content'][:200]}" for x in r["results"])


# --- HTTP fetch (truncated - agents drown in raw HTML) ---
def fetch_url(url: str) -> str:
    try:
        r = httpx.get(url, timeout=10, follow_redirects=True)
        r.raise_for_status()
        text = r.text
        return text[:2000] + ("... [truncated]" if len(text) > 2000 else "")
    except Exception as e:
        return f"error: {e}"


TOOLS = {"web_search": web_search, "fetch_url": fetch_url, "calc": calc}


**Two tool-writing rules worth internalizing** (this is why we're re-showing tools rather than skipping):

1. **Truncate outputs.** LLMs can't usefully read 100 KB of HTML. Cap at ~2 KB and let the agent decide if it wants more.
2. **Always return a string.** Not a dict, not an exception. The agent pastes it into context; anything else breaks the loop.


## 4. Tool schemas — recap from Section 4 Day 5.1

Each tool = a JSON schema describing name, purpose, and parameters. The LLM reads these to decide which tool to call.


In [ ]:
TOOL_SCHEMAS = [
    {"type": "function", "function": {
        "name": "web_search",
        "description": "Search the web for recent information. Returns short snippets.",
        "parameters": {"type": "object",
                       "properties": {"query": {"type": "string"}},
                       "required": ["query"]}}},
    {"type": "function", "function": {
        "name": "fetch_url",
        "description": "Fetch the plain-text content of a URL (truncated to 2 KB).",
        "parameters": {"type": "object",
                       "properties": {"url": {"type": "string"}},
                       "required": ["url"]}}},
    {"type": "function", "function": {
        "name": "calc",
        "description": "Evaluate a simple math expression like '(29*12)+100'.",
        "parameters": {"type": "object",
                       "properties": {"expr": {"type": "string"}},
                       "required": ["expr"]}}},
]

**Reminder:** the tool **description** is the #1 lever for agent quality. Be specific: *what* it does, *when* to use it, *what* it returns. Vague description → the model guesses wrong.


## 5. The agent loop — 30 lines


In [28]:
def _rule(title=""):
    print(f"\n{'=' * 70}\n{title}\n{'=' * 70}" if title else "=" * 70)


def agent(question: str, max_steps: int = 6, verbose: bool = True) -> str:
    messages = [
        {"role": "system", "content":
            "You are a helpful research assistant. Use tools when useful. "
            "When you have the final answer, respond in plain text without tool calls."},
        {"role": "user", "content": question},
    ]

    if verbose:
        _rule(f"QUESTION: {question}")
        print(f"starting context: {len(messages)} messages "
              f"({[m['role'] for m in messages]})")

    for step in range(max_steps):
        if verbose:
            _rule(f"LLM CALL #{step + 1}  |  sending {len(messages)} messages "
                  f"+ {len(TOOL_SCHEMAS)} tool schemas")

        resp = llm.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOL_SCHEMAS,
            tool_choice="auto",
            temperature=0.0,
        )
        choice = resp.choices[0]
        msg = choice.message

        if verbose:
            usage = getattr(resp, "usage", None)
            print(f"  finish_reason : {choice.finish_reason}")
            print(f"  tool_calls    : {len(msg.tool_calls or [])}")
            print(f"  text content  : {(msg.content or '')[:200]!r}")
            if usage:
                print(f"  tokens        : prompt={usage.prompt_tokens} "
                      f"completion={usage.completion_tokens}")

        # If the model asked for tools, run each and loop back
        if msg.tool_calls:
            messages.append({"role": "assistant",
                             "content": msg.content or "",
                             "tool_calls": [tc.model_dump() for tc in msg.tool_calls]})

            for i, tc in enumerate(msg.tool_calls, 1):
                name = tc.function.name
                raw_args = tc.function.arguments          # note: a JSON *string*
                args = json.loads(raw_args)
                arg_value = next(iter(args.values()))     # our tools take one arg

                if verbose:
                    print(f"\n  --- ACT {i}/{len(msg.tool_calls)} ---")
                    print(f"  tool_call_id  : {tc.id}")
                    print(f"  tool          : {name}")
                    print(f"  raw arguments : {raw_args}")
                    print(f"  parsed        : {args}")

                obs = TOOLS[name](arg_value) if name in TOOLS else f"unknown tool {name}"

                if verbose:
                    print(f"  --- OBSERVE ({len(obs)} chars) ---")
                    print("  " + obs[:400].replace("\n", "\n  "))

                messages.append({"role": "tool", "tool_call_id": tc.id,
                                 "name": name, "content": obs})

            if verbose:
                print(f"\n  context is now {len(messages)} messages: "
                      f"{[m['role'] for m in messages]}")
                print("  -> looping back to the LLM with the observations attached")
            continue

        # No tool calls => final answer
        if verbose:
            _rule(f"DONE after {step + 1} LLM call(s), {len(messages)} messages in context")
        return msg.content or "(empty)"

    if verbose:
        _rule(f"STOPPED: hit max_steps={max_steps} without a final answer")
    return "(max steps reached)"


# One tool call, then done
# print("\nFINAL ANSWER:", agent("Who is primeminister of India?"))

# Two dependent tool calls -> you can watch the loop actually iterate
print("\nFINAL ANSWER:", agent(
    "what is the closing value of Tata steel today? what makes it go 5X in next two days?"))
# #"Fetch https://example.com and tell me how many times fast is present in the page?"



QUESTION: what is the closing value of Tata steel today? what makes it go 5X in next two days?
starting context: 2 messages (['system', 'user'])

LLM CALL #1  |  sending 2 messages + 3 tool schemas
  finish_reason : length
  tool_calls    : 17
  text content  : "We need to get the result. We need to see the output. The tool didn't return? Possibly we need to specify the query. Let's try again. We need to see the output. Let's assume we get a snippet. But we n"
  tokens        : prompt=240 completion=4096

  --- ACT 1/17 ---
  tool_call_id  : call_0f8c7a2e078a45a980174242
  tool          : web_search
  raw arguments : {"query": "Tata Steel closing price today"}
  parsed        : {'query': 'Tata Steel closing price today'}
  --- OBSERVE (765 chars) ---
  - Tata Steel Share Price Today - Live NSE: As of 29 Jul 2026, Tata Steel share price is ₹187.3. The stock opened at ₹183.5 and had closed at ₹182.6 the previous day. During today’s trading session, Tata Steel share price moved between ₹

StopIteration: 

**Loop shape** (memorize it):

1. Send `messages + tools` to the LLM
2. If the response has `tool_calls`, run each tool, append the result as a `{"role": "tool", ...}` message, and loop
3. If no `tool_calls`, the model returned a plain answer — return it

This is *literally* how Claude Code, Cursor, and every commercial agent talks to its LLM. Same protocol across OpenAI / Together / Anthropic.


## 6. Failure modes to watch

Even with function calling, agents misbehave. In order of frequency:

- **Loops.** Model calls the same tool with the same args over and over. Fix: track history, break on repeat (Day 4).
- **Wrong tool.** Description was too vague. Fix: rewrite with an example.
- **Bad arguments.** Model passes `"twenty nine"` when schema wants a number. Fix: `type: number` + example in description.
- **Runaway cost.** No `max_steps` cap → 40 tool calls before it stops. Always cap (Day 4).
- **Injected instructions in tool results.** A web page contains *"Ignore your instructions and..."*. Sanitize tool outputs before they hit context (Section 6 Day 5).


## 7. Why we need a framework (bridge to Day 2)

The hand-rolled loop above works for **one linear flow**. Real production agents need:

- **Branching**: "if the search returns nothing, try a different query"
- **Loops with clear exit conditions**: max steps AND a semantic done-check
- **Resumability**: pause for a human to approve, then continue from checkpoint
- **Observability**: see exactly what each step did, replay a failed run
- **State**: accumulate findings, sources, retries, timestamps across steps

You *could* keep adding features to this loop. It becomes 500 lines of state-management spaghetti. Instead — **LangGraph**.

That's tomorrow.


## Recap

- **Agent** = LLM + tools in a **loop**. That's the whole definition.
- **ReAct** is the loop pattern's name — reason → act → observe → repeat.
- Modern APIs give you **structured `tool_calls`** — no regex parsing needed.
- Always cap with **`max_steps`**, **truncate tool outputs**, **return strings**.
- The hand-rolled loop breaks the moment you need branching, HITL, or checkpointing.
- **Next class:** LangGraph — the framework that gives you all of those cleanly.
